# 02_pdf_crawler — 국정감사 회의록(문화체육) PDF 수집 파이프라인

**Stage**: `PDF_CRAWL_AND_DOWNLOAD`
**Scope**: Excel Registry → PDF 원본 다운로드 → 무결성 검증(signature/open/page_count/SHA-256) → checkpoint → manifest/report까지.
**이 노트북이 하지 않는 것**: PDF 본문 텍스트 추출, OCR, 페이지/블록 분할, 발언자 판별, TF-IDF, embedding, SQLite 미러링.
**Required branch**: `P3_DATA_RAW`

다른 notebook이나 외부 `.py` 모듈을 import하지 않는다. 모든 상수·함수·실행·export는 이 노트북 안에서 완결된다.


## 00. 실행 계약 · 환경 · 경로

### 계약
- 직접 입력: `data_origin/국정감사회의록_문화체육 2020~.xlsx`
- 직접 산출물: `pdf_raw_data/{meeting_id}.pdf`, `data_parse/pdf_crawler/control_registry.parquet`,
  `download_log.parquet`, `crawl_quality.parquet`, `pipeline_manifest.json`, `CRAWL_REPORT.md`,
  `logs/pdf_crawler.log`, `data_dict/02_pdf_crawler_data_dictionary.md`
- 금지: Selenium/Playwright, Registry 외 URL 요청, 401/403 우회, CAPTCHA 우회,
  `page.get_text()` / `page.get_text("blocks")` / `page.get_pixmap()` 호출.
  PyMuPDF/pypdf는 이 단계에서 오직 `fitz.open()` + `page_count` 확인에만 사용한다.


In [1]:
import sys, os, subprocess, platform, json, hashlib, re, time, logging
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlparse

import pandas as pd
import numpy as np
import requests
import fitz  # PyMuPDF -- this stage only uses fitz.open() + doc.page_count
import pyarrow
import nbformat

REPO_ROOT = Path.cwd()
assert (REPO_ROOT / "SSOT.md").exists() and (REPO_ROOT / "data_origin").exists(), (
    f"Repository markers (SSOT.md, data_origin/) not found under {REPO_ROOT}. "
    "This notebook must be executed with its own directory as the working directory."
)

EXCEL_PATH = REPO_ROOT / "data_origin" / "국정감사회의록_문화체육 2020~.xlsx"
PDF_RAW_DIR = REPO_ROOT / "pdf_raw_data"
CRAWLER_OUT_DIR = REPO_ROOT / "data_parse" / "pdf_crawler"
LOG_DIR = CRAWLER_OUT_DIR / "logs"
DATA_DICT_DIR = REPO_ROOT / "data_dict"

for d in (PDF_RAW_DIR, CRAWLER_OUT_DIR, LOG_DIR, DATA_DICT_DIR):
    d.mkdir(parents=True, exist_ok=True)

CONTROL_REGISTRY_PATH = CRAWLER_OUT_DIR / "control_registry.parquet"
DOWNLOAD_LOG_PATH = CRAWLER_OUT_DIR / "download_log.parquet"
CRAWL_QUALITY_PATH = CRAWLER_OUT_DIR / "crawl_quality.parquet"
MANIFEST_PATH = CRAWLER_OUT_DIR / "pipeline_manifest.json"
CRAWL_REPORT_PATH = CRAWLER_OUT_DIR / "CRAWL_REPORT.md"
DATA_DICT_PATH = DATA_DICT_DIR / "02_pdf_crawler_data_dictionary.md"
LOG_PATH = LOG_DIR / "pdf_crawler.log"

STAGE = "PDF_CRAWL_AND_DOWNLOAD"
EXECUTION_CONTRACT_STATUS = "CRAWLER_EXECUTION_CONTRACT_ACTIVE"
RUN_STARTED_AT = datetime.now(timezone.utc).isoformat()

logger = logging.getLogger("pdf_crawler")
logger.setLevel(logging.INFO)
logger.handlers.clear()
_fh = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8")
_fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
logger.addHandler(_fh)
_sh = logging.StreamHandler()
_sh.setFormatter(logging.Formatter("%(levelname)s %(message)s"))
logger.addHandler(_sh)

def _git(*args):
    try:
        return subprocess.run(["git", *args], cwd=REPO_ROOT, capture_output=True,
                               text=True, check=True).stdout.strip()
    except Exception as e:
        logger.warning(f"git command failed: {args} ({e})")
        return None

GIT_BRANCH = _git("branch", "--show-current")
GIT_COMMIT = _git("rev-parse", "HEAD")

if GIT_BRANCH is not None:
    assert GIT_BRANCH == "P3_DATA_RAW", f"Required branch P3_DATA_RAW, found {GIT_BRANCH!r}"

ENV_INFO = {
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "pyarrow": pyarrow.__version__,
    "requests": requests.__version__,
    "pymupdf": getattr(fitz, "pymupdf_version", getattr(fitz, "__version__", "unknown")),
    "nbformat": nbformat.__version__,
    "git_branch": GIT_BRANCH,
    "git_commit": GIT_COMMIT,
}

logger.info(f"STAGE={STAGE} contract_status={EXECUTION_CONTRACT_STATUS}")
logger.info(f"repo_root={REPO_ROOT}")
logger.info(f"env={ENV_INFO}")

for k, v in ENV_INFO.items():
    print(f"{k}: {v}")


INFO STAGE=PDF_CRAWL_AND_DOWNLOAD contract_status=CRAWLER_EXECUTION_CONTRACT_ACTIVE


INFO repo_root=/home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE


INFO env={'python_version': '3.12.3', 'platform': 'Linux-6.18.33.2-microsoft-standard-WSL2-x86_64-with-glibc2.39', 'pandas': '3.0.3', 'pyarrow': '25.0.0', 'requests': '2.31.0', 'pymupdf': '1.28.0', 'nbformat': '5.10.4', 'git_branch': 'P3_DATA_RAW', 'git_commit': '236ad55872afe97df3c9308f9fbd3535f5d287be'}


python_version: 3.12.3
platform: Linux-6.18.33.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
pandas: 3.0.3
pyarrow: 25.0.0
requests: 2.31.0
pymupdf: 1.28.0
nbformat: 5.10.4
git_branch: P3_DATA_RAW
git_commit: 236ad55872afe97df3c9308f9fbd3535f5d287be


## 01. 입력 Excel Registry 로드

파일 무결성(존재/크기/SHA-256)과 실제 sheet/컬럼 구조를 먼저 감사한다. 이 감사 없이는 어떤 파싱도 진행하지 않는다.


In [2]:
def sha256_of_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def audit_excel_registry(path: Path) -> dict:
    assert path.exists(), f"input excel not found: {path}"
    audit = {
        "path": str(path.relative_to(REPO_ROOT)),
        "exists": True,
        "size_bytes": path.stat().st_size,
        "sha256": sha256_of_file(path),
    }
    xls = pd.ExcelFile(path)
    audit["sheet_names"] = xls.sheet_names
    sheet_reports = {}
    for sn in xls.sheet_names:
        df = pd.read_excel(path, sheet_name=sn, dtype=str)
        sheet_reports[sn] = {
            "n_rows": len(df),
            "n_cols": len(df.columns),
            "columns": list(df.columns),
            "fully_blank_rows": int(df.isna().all(axis=1).sum()),
            "fully_blank_cols": [c for c in df.columns if df[c].isna().all()],
        }
    audit["sheets"] = sheet_reports
    return audit


In [3]:
registry_audit = audit_excel_registry(EXCEL_PATH)
logger.info(f"registry_audit_top={ {k:v for k,v in registry_audit.items() if k!='sheets'} }")
for sn, rep in registry_audit["sheets"].items():
    logger.info(f"sheet={sn} rows={rep['n_rows']} cols={rep['n_cols']} columns={rep['columns']}")

print(json.dumps(registry_audit, ensure_ascii=False, indent=2))


INFO registry_audit_top={'path': 'data_origin/국정감사회의록_문화체육 2020~.xlsx', 'exists': True, 'size_bytes': 20568, 'sha256': 'a4e7bb5ae119556e9c506d74cc0566caaf59a753f3f2c26cb30004ad486ea675', 'sheet_names': ['Sheet1']}


INFO sheet=Sheet1 rows=42 cols=9 columns=['회의ID', '대수', '회기', '차수', '회의일자', '회의종류', '위원회코드', '위원회명', '다운로드 URL']


{
  "path": "data_origin/국정감사회의록_문화체육 2020~.xlsx",
  "exists": true,
  "size_bytes": 20568,
  "sha256": "a4e7bb5ae119556e9c506d74cc0566caaf59a753f3f2c26cb30004ad486ea675",
  "sheet_names": [
    "Sheet1"
  ],
  "sheets": {
    "Sheet1": {
      "n_rows": 42,
      "n_cols": 9,
      "columns": [
        "회의ID",
        "대수",
        "회기",
        "차수",
        "회의일자",
        "회의종류",
        "위원회코드",
        "위원회명",
        "다운로드 URL"
      ],
      "fully_blank_rows": 0,
      "fully_blank_cols": []
    }
  }
}


## 02. Sheet 선택, 컬럼 정규화 및 Control Table 생성

사용할 sheet는 회의ID 계열 컬럼과 PDF URL 계열 컬럼을 모두 가진 sheet다. 조건을 만족하는 sheet가
정확히 1개가 아니면 `AMBIGUOUS_REGISTRY_SHEET`로 즉시 중단한다.

Canonical 컬럼으로 매핑되지 않는 원본 컬럼은 삭제하지 않고 `source_{정규화된_컬럼명}`으로 보존한다.
`meeting_id/meeting_number/meeting_count/meeting_year/meeting_date`는 숫자로 캐스팅하지 않는다(SSOT.md §4.1).
이번 단계의 모든 `parse_status` 계열 값은 `pending`/`pd.NA`로 고정하며 PDF 다운로드 여부와 무관하게 변경하지 않는다.


In [4]:
MEETING_ID_COL_CANDIDATES = ["회의ID", "회의 ID", "meeting_id"]
URL_COL_CANDIDATES = ["다운로드 URL", "다운로드URL", "PDF URL", "source_url"]

def select_registry_sheet(audit: dict) -> str:
    candidates = []
    for sn, rep in audit["sheets"].items():
        cols = rep["columns"]
        has_id = any(c in cols for c in MEETING_ID_COL_CANDIDATES)
        has_url = any(c in cols for c in URL_COL_CANDIDATES)
        if has_id and has_url:
            candidates.append(sn)
    if len(candidates) != 1:
        raise RuntimeError(f"AMBIGUOUS_REGISTRY_SHEET candidates={candidates}")
    return candidates[0]

SELECTED_SHEET = select_registry_sheet(registry_audit)
logger.info(f"selected_sheet={SELECTED_SHEET}")

raw_df = pd.read_excel(EXCEL_PATH, sheet_name=SELECTED_SHEET, dtype=str)
ID_COL = next(c for c in MEETING_ID_COL_CANDIDATES if c in raw_df.columns)
URL_COL = next(c for c in URL_COL_CANDIDATES if c in raw_df.columns)
print(f"selected_sheet={SELECTED_SHEET!r} id_col={ID_COL!r} url_col={URL_COL!r} rows={len(raw_df)}")


INFO selected_sheet=Sheet1


selected_sheet='Sheet1' id_col='회의ID' url_col='다운로드 URL' rows=42


In [5]:
def normalize_col_name(c: str) -> str:
    return re.sub(r"\s+", "_", c.strip())

def _extract_leading_int(s):
    if pd.isna(s):
        return pd.NA
    m = re.search(r"\d+", str(s))
    return m.group(0) if m else pd.NA

def build_canonical_registry(df: pd.DataFrame) -> pd.DataFrame:
    df = df.reset_index(drop=True)
    n = len(df)
    out = pd.DataFrame(index=df.index)

    out["meeting_id"] = df[ID_COL].astype(str).str.strip()
    out["meeting_number"] = df.get("대수", pd.Series([pd.NA] * n)).map(_extract_leading_int)
    out["meeting_count"] = df.get("회기", pd.Series([pd.NA] * n)).map(_extract_leading_int)

    dates = pd.to_datetime(df.get("회의일자"), errors="coerce")
    out["meeting_year"] = dates.dt.strftime("%y")
    out["meeting_date"] = dates.dt.strftime("%m%d")
    out.loc[dates.isna(), "meeting_year"] = pd.NA
    out.loc[dates.isna(), "meeting_date"] = pd.NA

    out["meeting_type"] = df.get("회의종류")
    out["committee_code"] = df.get("위원회코드")
    out["committee_name"] = df.get("위원회명")
    out["source_url"] = df[URL_COL].astype(str).str.strip()
    out["source_row_no"] = pd.array(df.index + 1, dtype="Int64")  # 1-based, header excluded

    handled_original_cols = {ID_COL, URL_COL, "대수", "회기", "회의일자", "회의종류", "위원회코드", "위원회명"}
    for c in df.columns:
        if c not in handled_original_cols:
            out[f"source_{normalize_col_name(c)}"] = df[c]

    # download-state columns (pending until Phase 08 resolves them)
    out["download_status"] = "pending"
    out["http_status"] = pd.array([pd.NA] * n, dtype="Int64")
    out["final_url"] = pd.NA
    out["local_pdf_path"] = pd.NA
    out["file_size"] = pd.array([pd.NA] * n, dtype="Int64")
    out["sha256"] = pd.NA
    out["page_count"] = pd.array([pd.NA] * n, dtype="Int64")
    out["reused_existing_file"] = False
    out["error_message"] = pd.NA

    # parse-state columns: untouched in this stage regardless of download outcome
    out["parse_status"] = "pending"
    out["parser_name"] = pd.NA
    out["parser_version"] = pd.NA
    out["extracted_char_count"] = pd.array([pd.NA] * n, dtype="Int64")
    out["ocr_page_count"] = pd.array([pd.NA] * n, dtype="Int64")

    return out

control_registry = build_canonical_registry(raw_df)
logger.info(f"control_registry rows={len(control_registry)} cols={list(control_registry.columns)}")
control_registry.head(10)


INFO control_registry rows=42 cols=['meeting_id', 'meeting_number', 'meeting_count', 'meeting_year', 'meeting_date', 'meeting_type', 'committee_code', 'committee_name', 'source_url', 'source_row_no', 'source_차수', 'download_status', 'http_status', 'final_url', 'local_pdf_path', 'file_size', 'sha256', 'page_count', 'reused_existing_file', 'error_message', 'parse_status', 'parser_name', 'parser_version', 'extracted_char_count', 'ocr_page_count']


,meeting_id,meeting_number,meeting_count,meeting_year,meeting_date,meeting_type,committee_code,committee_name,source_url,source_row_no,...,file_size,sha256,page_count,reused_existing_file,error_message,parse_status,parser_name,parser_version,extracted_char_count,ocr_page_count
0,N053487,22,429,25,1029,국정감사 회의록,AL,문화체육관광위원회,https://record.assembly.go.kr/assembly/viewer/...,1,...,<NA>,<NA>,<NA>,False,<NA>,pending,<NA>,<NA>,<NA>,<NA>
1,N053485,22,429,25,1027,국정감사 회의록,AL,문화체육관광위원회,https://record.assembly.go.kr/assembly/viewer/...,2,...,<NA>,<NA>,<NA>,False,<NA>,pending,<NA>,<NA>,<NA>,<NA>
2,N053483,22,429,25,1023,국정감사 회의록,AL,문화체육관광위원회,https://record.assembly.go.kr/assembly/viewer/...,3,...,<NA>,<NA>,<NA>,False,<NA>,pending,<NA>,<NA>,<NA>,<NA>
3,N053482,22,429,25,1022,국정감사 회의록,AL,문화체육관광위원회,https://record.assembly.go.kr/assembly/viewer/...,4,...,<NA>,<NA>,<NA>,False,<NA>,pending,<NA>,<NA>,<NA>,<NA>
4,N053480,22,429,25,1020,국정감사 회의록,AL,문화체육관광위원회,https://record.assembly.go.kr/assembly/viewer/...,5,...,<NA>,<NA>,<NA>,False,<NA>,pending,<NA>,<NA>,<NA>,<NA>
5,N053481,22,429,25,1016,국정감사 회의록,AL,문화체육관광위원회,https://record.assembly.go.kr/assembly/viewer/...,6,...,<NA>,<NA>,<NA>,False,<NA>,pending,<NA>,<NA>,<NA>,<NA>
6,N053479,22,429,25,1014,국정감사 회의록,AL,문화체육관광위원회,https://record.assembly.go.kr/assembly/viewer/...,7,...,<NA>,<NA>,<NA>,False,<NA>,pending,<NA>,<NA>,<NA>,<NA>
7,054622,22,418,24,1024,국정감사 회의록,AL,문화체육관광위원회,https://record.assembly.go.kr/assembly/viewer/...,8,...,<NA>,<NA>,<NA>,False,<NA>,pending,<NA>,<NA>,<NA>,<NA>
8,054576,22,418,24,1022,국정감사 회의록,AL,문화체육관광위원회,https://record.assembly.go.kr/assembly/viewer/...,9,...,<NA>,<NA>,<NA>,False,<NA>,pending,<NA>,<NA>,<NA>,<NA>
9,054519,22,418,24,1018,국정감사 회의록,AL,문화체육관광위원회,https://record.assembly.go.kr/assembly/viewer/...,10,...,<NA>,<NA>,<NA>,False,<NA>,pending,<NA>,<NA>,<NA>,<NA>


## 03. Registry 품질검사 (Critical Gate)

다음 중 하나라도 발견되면 `critical_gate_failed=True`로 표시하고, 전체 게이트 상태가 `FAILED`이면
이후 어떤 신규 다운로드도 실행하지 않는다: meeting_id 결측/중복(경로충돌), source_url 결측,
http/https 이외 스킴, meeting_id 파일명 안전성 실패, 예상 PDF 경로 충돌.

서로 다른 meeting_id가 동일 source_url을 쓰는 경우는 하드 게이트 실패가 아니라 해당 행만
`DUPLICATE_SOURCE_URL_ACROSS_MEETING_IDS`로 보류(pending)하고 나머지 독립 URL은 계속 진행한다.


In [6]:
FILENAME_SAFE_RE = re.compile(r"^[A-Za-z0-9_-]+$")
URL_SCHEME_RE = re.compile(r"^https?://", re.IGNORECASE)

CRITICAL_GATE_REPORT = {"checks": [], "status": "UNKNOWN"}

def _record_check(name, failed_idx, detail=""):
    CRITICAL_GATE_REPORT["checks"].append(
        {"check": name, "failed_count": int(len(failed_idx)), "detail": detail}
    )

def run_critical_gate(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    critical_mask = pd.Series(False, index=df.index)

    missing_id = df["meeting_id"].isna() | (df["meeting_id"].astype(str).str.strip() == "")
    _record_check("MISSING_MEETING_ID", df.index[missing_id])
    critical_mask |= missing_id

    dup_id = df["meeting_id"].duplicated(keep=False) & ~missing_id
    _record_check("DUPLICATE_MEETING_ID_PATH_COLLISION", df.index[dup_id])
    critical_mask |= dup_id

    missing_url = df["source_url"].isna() | (df["source_url"].astype(str).str.strip() == "")
    _record_check("MISSING_SOURCE_URL", df.index[missing_url])
    critical_mask |= missing_url

    bad_scheme = (~df["source_url"].astype(str).str.match(URL_SCHEME_RE)) & (~missing_url)
    _record_check("NON_HTTP_SCHEME_URL", df.index[bad_scheme])
    critical_mask |= bad_scheme

    unsafe_filename = (~df["meeting_id"].astype(str).str.match(FILENAME_SAFE_RE)) & (~missing_id)
    _record_check("UNSAFE_MEETING_ID_FILENAME", df.index[unsafe_filename])
    critical_mask |= unsafe_filename

    expected_path = df["meeting_id"].astype(str).map(lambda m: f"pdf_raw_data/{m}.pdf")
    path_collision = expected_path.duplicated(keep=False) & (~missing_id)
    _record_check("EXPECTED_PDF_PATH_COLLISION", df.index[path_collision])
    critical_mask |= path_collision

    url_dup_mask = df["source_url"].duplicated(keep=False) & (~missing_url)
    cross_id_dup = pd.Series(False, index=df.index)
    if url_dup_mask.any():
        multi = df.loc[url_dup_mask].groupby("source_url")["meeting_id"].transform(lambda s: s.nunique() > 1)
        cross_id_dup.loc[url_dup_mask] = multi.values
    _record_check("DUPLICATE_SOURCE_URL_ACROSS_MEETING_IDS", df.index[cross_id_dup],
                  "row-level pending, not a hard gate failure")

    df["critical_gate_failed"] = critical_mask
    df.loc[cross_id_dup, "download_status"] = "pending"
    df.loc[cross_id_dup, "error_message"] = "DUPLICATE_SOURCE_URL_ACROSS_MEETING_IDS"

    n_critical = int(critical_mask.sum())
    CRITICAL_GATE_REPORT["status"] = "FAILED" if n_critical > 0 else "PASSED"
    CRITICAL_GATE_REPORT["n_critical_failed_rows"] = n_critical
    CRITICAL_GATE_REPORT["n_total_rows"] = len(df)
    return df

control_registry = run_critical_gate(control_registry)
logger.info(f"critical_gate={CRITICAL_GATE_REPORT}")
print(json.dumps(CRITICAL_GATE_REPORT, ensure_ascii=False, indent=2))
if CRITICAL_GATE_REPORT["status"] == "FAILED":
    logger.error("Critical gate FAILED -- no HTTP download will be executed this run.")


INFO critical_gate={'checks': [{'check': 'MISSING_MEETING_ID', 'failed_count': 0, 'detail': ''}, {'check': 'DUPLICATE_MEETING_ID_PATH_COLLISION', 'failed_count': 0, 'detail': ''}, {'check': 'MISSING_SOURCE_URL', 'failed_count': 0, 'detail': ''}, {'check': 'NON_HTTP_SCHEME_URL', 'failed_count': 0, 'detail': ''}, {'check': 'UNSAFE_MEETING_ID_FILENAME', 'failed_count': 0, 'detail': ''}, {'check': 'EXPECTED_PDF_PATH_COLLISION', 'failed_count': 0, 'detail': ''}, {'check': 'DUPLICATE_SOURCE_URL_ACROSS_MEETING_IDS', 'failed_count': 0, 'detail': 'row-level pending, not a hard gate failure'}], 'status': 'PASSED', 'n_critical_failed_rows': 0, 'n_total_rows': 42}


{
  "checks": [
    {
      "check": "MISSING_MEETING_ID",
      "failed_count": 0,
      "detail": ""
    },
    {
      "check": "DUPLICATE_MEETING_ID_PATH_COLLISION",
      "failed_count": 0,
      "detail": ""
    },
    {
      "check": "MISSING_SOURCE_URL",
      "failed_count": 0,
      "detail": ""
    },
    {
      "check": "NON_HTTP_SCHEME_URL",
      "failed_count": 0,
      "detail": ""
    },
    {
      "check": "UNSAFE_MEETING_ID_FILENAME",
      "failed_count": 0,
      "detail": ""
    },
    {
      "check": "EXPECTED_PDF_PATH_COLLISION",
      "failed_count": 0,
      "detail": ""
    },
    {
      "check": "DUPLICATE_SOURCE_URL_ACROSS_MEETING_IDS",
      "failed_count": 0,
      "detail": "row-level pending, not a hard gate failure"
    }
  ],
  "status": "PASSED",
  "n_critical_failed_rows": 0,
  "n_total_rows": 42
}


## 04. 기존 PDF 검사 및 다운로드 계획

기존 `pdf_raw_data/{meeting_id}.pdf`가 있으면 크기>0, `%PDF-` signature, PDF open 성공,
`page_count>=1`, SHA-256 계산을 모두 통과할 때만 재사용한다. 하나라도 손상되어 있으면
`EXISTING_PDF_INVALID` + circuit breaker로 **전체 신규 다운로드를 중단**한다(해당 행만 건너뛰지 않는다).


In [7]:
def validate_existing_pdf(path: Path) -> dict:
    result = {"valid": False, "file_size": None, "sha256": None, "page_count": None, "error_message": None}
    try:
        size = path.stat().st_size
        result["file_size"] = size
        if size <= 0:
            result["error_message"] = "EMPTY_FILE"
            return result
        with open(path, "rb") as f:
            head = f.read(5)
        if not head.startswith(b"%PDF-"):
            result["error_message"] = "BAD_PDF_SIGNATURE"
            return result
        doc = fitz.open(path)
        page_count = doc.page_count
        doc.close()
        if page_count < 1:
            result["error_message"] = "ZERO_PAGE_COUNT"
            return result
        result["page_count"] = page_count
        result["sha256"] = sha256_of_file(path)
        result["valid"] = True
    except Exception as e:
        result["error_message"] = f"OPEN_FAILED:{type(e).__name__}:{e}"
    return result

EXISTING_PDF_CIRCUIT_BREAKER = {"triggered": False, "meeting_id": None, "error_message": None}

def check_existing_pdfs(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for idx, row in df.iterrows():
        if row["critical_gate_failed"]:
            continue
        meeting_id = row["meeting_id"]
        expected_path = PDF_RAW_DIR / f"{meeting_id}.pdf"
        if not expected_path.exists():
            continue
        chk = validate_existing_pdf(expected_path)
        if chk["valid"]:
            df.at[idx, "download_status"] = "downloaded"
            df.at[idx, "reused_existing_file"] = True
            df.at[idx, "local_pdf_path"] = str(expected_path.relative_to(REPO_ROOT))
            df.at[idx, "file_size"] = chk["file_size"]
            df.at[idx, "sha256"] = chk["sha256"]
            df.at[idx, "page_count"] = chk["page_count"]
            logger.info(f"existing PDF reused meeting_id={meeting_id}")
        else:
            df.at[idx, "download_status"] = "failed"
            df.at[idx, "error_message"] = "EXISTING_PDF_INVALID"
            EXISTING_PDF_CIRCUIT_BREAKER.update(
                triggered=True, meeting_id=meeting_id, error_message=chk["error_message"]
            )
            logger.error(f"EXISTING_PDF_INVALID meeting_id={meeting_id} detail={chk['error_message']}")
            break
    return df

control_registry = check_existing_pdfs(control_registry)
print(f"circuit_breaker={EXISTING_PDF_CIRCUIT_BREAKER}")


circuit_breaker={'triggered': False, 'meeting_id': None, 'error_message': None}


In [8]:
def build_download_plan(df: pd.DataFrame) -> pd.DataFrame:
    eligible = (
        (~df["critical_gate_failed"])
        & (df["download_status"] == "pending")
        & (df["error_message"].isna())
    )
    return df.loc[eligible].copy()

if EXISTING_PDF_CIRCUIT_BREAKER["triggered"]:
    download_plan = control_registry.iloc[0:0].copy()
    logger.error("Download plan EMPTY: circuit breaker triggered on existing PDF corruption.")
elif CRITICAL_GATE_REPORT["status"] == "FAILED":
    download_plan = control_registry.iloc[0:0].copy()
    logger.error("Download plan EMPTY: Registry Critical Gate FAILED.")
else:
    download_plan = build_download_plan(control_registry)

print(f"download plan rows: {len(download_plan)} / total {len(control_registry)}")


download plan rows: 42 / total 42


## 05. HTTP Session · Retry · Rate Limit

고정 정책: 순차 다운로드(`MAX_WORKERS=1`), 요청 간격 1.5초, 최대 재시도 4회, backoff `[2,4,8,16]`초,
connect timeout 10초 / read timeout 180초, chunk 1MiB, redirect 허용, User-Agent 고정.
Registry의 `source_url` 외 어떤 URL도 요청하지 않는다.


In [9]:
MAX_WORKERS = 1
REQUEST_INTERVAL_SECONDS = 1.5
MAX_RETRIES = 4
CONNECT_TIMEOUT_SECONDS = 10
READ_TIMEOUT_SECONDS = 180
CHUNK_SIZE_BYTES = 1024 * 1024
ALLOW_REDIRECTS = True
USER_AGENT = "SBS-DSJA-P3-Culture-PDF-Crawler/1.0"
RETRY_BACKOFF_SECONDS = [2, 4, 8, 16]

def create_session() -> requests.Session:
    s = requests.Session()
    s.headers.update({"User-Agent": USER_AGENT})
    return s

def classify_status_action(status_code: int) -> str:
    if status_code in (200, 206):
        return "validate"
    if status_code == 400:
        return "failed"
    if status_code in (401, 403):
        return "failed"
    if status_code in (404, 410):
        return "invalid_url"
    if status_code == 408:
        return "retry"
    if status_code == 429:
        return "retry_after"
    if status_code in (500, 502, 503, 504):
        return "retry_backoff"
    return "failed"

SESSION = create_session()
print("session created; headers:", dict(SESSION.headers))


session created; headers: {'User-Agent': 'SBS-DSJA-P3-Culture-PDF-Crawler/1.0', 'Accept-Encoding': 'gzip, deflate', 'Accept': '*/*', 'Connection': 'keep-alive'}


## 06. PDF 스트리밍 다운로드

`.part` 파일에 chunk 단위로 스트리밍 저장한다. 검증 전에는 최종 `.pdf` 파일명을 사용하지 않는다.
429는 `Retry-After` 헤더를 우선 사용하고, 그 외 재시도 대상은 고정 backoff를 사용한다.


In [10]:
def attempt_download(session: requests.Session, url: str, part_path: Path) -> dict:
    attempt = {
        "url_requested": url, "http_status": None, "final_url": None,
        "bytes_received": 0, "action": None, "error_message": None,
        "elapsed_seconds": None, "retry_after_seconds": None,
    }
    t0 = time.monotonic()
    try:
        resp = session.get(
            url, stream=True, allow_redirects=ALLOW_REDIRECTS,
            timeout=(CONNECT_TIMEOUT_SECONDS, READ_TIMEOUT_SECONDS),
        )
    except requests.exceptions.RequestException as e:
        attempt["action"] = "retry_backoff"
        attempt["error_message"] = f"{type(e).__name__}:{e}"
        attempt["elapsed_seconds"] = time.monotonic() - t0
        return attempt

    attempt["http_status"] = resp.status_code
    attempt["final_url"] = resp.url
    action = classify_status_action(resp.status_code)
    attempt["action"] = action

    if action == "validate":
        try:
            total = 0
            with open(part_path, "wb") as f:
                for chunk in resp.iter_content(chunk_size=CHUNK_SIZE_BYTES):
                    if chunk:
                        f.write(chunk)
                        total += len(chunk)
            attempt["bytes_received"] = total
        except Exception as e:
            attempt["action"] = "failed"
            attempt["error_message"] = f"STREAM_WRITE_FAILED:{type(e).__name__}:{e}"
    elif action == "retry_after":
        ra = resp.headers.get("Retry-After")
        try:
            attempt["retry_after_seconds"] = float(ra) if ra is not None else None
        except ValueError:
            attempt["retry_after_seconds"] = None
        attempt["error_message"] = f"HTTP_{resp.status_code}"
    else:
        # non-validate outcomes: never persist the response body
        attempt["error_message"] = f"HTTP_{resp.status_code}"

    resp.close()
    attempt["elapsed_seconds"] = time.monotonic() - t0
    return attempt

print("attempt_download() defined")


attempt_download() defined


## 07. PDF 무결성 · SHA-256 · Page Count 검증

`%PDF-` signature, HTML/JSON 오류 본문 여부, `fitz.open()` 성공, `page_count>=1`, SHA-256을 검사한다.


In [11]:
def validate_downloaded_part(part_path: Path) -> dict:
    result = {"valid": False, "page_count": None, "sha256": None, "error_message": None}
    try:
        size = part_path.stat().st_size
        if size <= 0:
            result["error_message"] = "EMPTY_RESPONSE_BODY"
            return result
        with open(part_path, "rb") as f:
            head = f.read(1024)
        if not head.startswith(b"%PDF-"):
            low = head.lower()
            if b"<html" in low or b"<!doctype" in low:
                result["error_message"] = "HTML_ERROR_PAGE"
            elif head.strip()[:1] in (b"{", b"["):
                result["error_message"] = "JSON_ERROR_BODY"
            else:
                result["error_message"] = "NON_PDF_BODY"
            return result
        doc = fitz.open(part_path)
        page_count = doc.page_count
        doc.close()
        if page_count < 1:
            result["error_message"] = "ZERO_PAGE_COUNT"
            return result
        result["page_count"] = page_count
        result["sha256"] = sha256_of_file(part_path)
        result["valid"] = True
    except Exception as e:
        result["error_message"] = f"PDF_OPEN_FAILED:{type(e).__name__}:{e}"
    return result

print("validate_downloaded_part() defined")


validate_downloaded_part() defined


## 08. Checkpoint 및 구조화 산출물 저장

`CHECKPOINT_EVERY_N_RECORDS=1` — Registry 각 행의 상태가 결정될 때마다 즉시 원자적으로 checkpoint한다
(`*.parquet.tmp` → round-trip 검증 → `os.replace`). 이 셀이 실제 순차 다운로드 루프를 실행한다.


In [12]:
CHECKPOINT_EVERY_N_RECORDS = 1

def atomic_write_parquet(df: pd.DataFrame, target: Path) -> None:
    tmp = target.with_suffix(target.suffix + ".tmp")
    df.to_parquet(tmp, index=False)
    check = pd.read_parquet(tmp)
    assert len(check) == len(df), "parquet round-trip row count mismatch"
    os.replace(tmp, target)

DOWNLOAD_LOG_COLUMNS = [
    "meeting_id", "attempt_no", "timestamp", "url_requested", "http_status",
    "final_url", "bytes_received", "action", "error_message",
    "elapsed_seconds", "retry_after_seconds",
]
download_log_rows = []

def process_one_row(session: requests.Session, row: pd.Series) -> dict:
    meeting_id = row["meeting_id"]
    url = row["source_url"]
    final_pdf_path = PDF_RAW_DIR / f"{meeting_id}.pdf"
    part_path = PDF_RAW_DIR / f"{meeting_id}.pdf.part"

    update = {"meeting_id": meeting_id}
    backoff_idx = 0
    for attempt_no in range(1, MAX_RETRIES + 2):
        attempt = attempt_download(session, url, part_path)
        attempt["meeting_id"] = meeting_id
        attempt["attempt_no"] = attempt_no
        attempt["timestamp"] = datetime.now(timezone.utc).isoformat()
        download_log_rows.append(attempt)
        logger.info(
            f"meeting_id={meeting_id} attempt={attempt_no} "
            f"status={attempt['http_status']} action={attempt['action']}"
        )

        action = attempt["action"]
        if action == "validate":
            vres = validate_downloaded_part(part_path)
            if vres["valid"]:
                os.replace(part_path, final_pdf_path)
                update.update({
                    "download_status": "downloaded",
                    "http_status": attempt["http_status"],
                    "final_url": attempt["final_url"],
                    "local_pdf_path": str(final_pdf_path.relative_to(REPO_ROOT)),
                    "file_size": final_pdf_path.stat().st_size,
                    "sha256": vres["sha256"],
                    "page_count": vres["page_count"],
                    "error_message": pd.NA,
                })
            else:
                if part_path.exists():
                    part_path.unlink()
                update.update({
                    "download_status": "invalid_content",
                    "http_status": attempt["http_status"],
                    "final_url": attempt["final_url"],
                    "error_message": vres["error_message"],
                })
            return update

        if action == "invalid_url":
            update.update({"download_status": "invalid_url",
                            "http_status": attempt["http_status"],
                            "error_message": attempt["error_message"]})
            return update

        if action == "failed":
            update.update({"download_status": "failed",
                            "http_status": attempt["http_status"],
                            "error_message": attempt["error_message"]})
            return update

        # retry / retry_after / retry_backoff
        if attempt_no > MAX_RETRIES:
            update.update({"download_status": "failed",
                            "http_status": attempt["http_status"],
                            "error_message": attempt["error_message"] or "MAX_RETRIES_EXCEEDED"})
            return update

        if action == "retry_after" and attempt["retry_after_seconds"] is not None:
            wait_s = attempt["retry_after_seconds"]
        else:
            wait_s = RETRY_BACKOFF_SECONDS[min(backoff_idx, len(RETRY_BACKOFF_SECONDS) - 1)]
        time.sleep(wait_s)
        backoff_idx += 1

    update.update({"download_status": "failed", "http_status": None,
                    "error_message": "MAX_RETRIES_EXCEEDED"})
    return update


In [13]:
if EXISTING_PDF_CIRCUIT_BREAKER["triggered"]:
    logger.error("Skipping download loop: circuit breaker triggered on existing PDF corruption.")
elif CRITICAL_GATE_REPORT["status"] == "FAILED":
    logger.error("Skipping download loop: Registry Critical Gate FAILED.")
else:
    plan_indices = list(download_plan.index)
    for i, idx in enumerate(plan_indices):
        row = control_registry.loc[idx]
        result = process_one_row(SESSION, row)
        for k, v in result.items():
            if k == "meeting_id":
                continue
            control_registry.at[idx, k] = v
        if (i + 1) % CHECKPOINT_EVERY_N_RECORDS == 0:
            atomic_write_parquet(control_registry, CONTROL_REGISTRY_PATH)
            log_df = pd.DataFrame(download_log_rows, columns=DOWNLOAD_LOG_COLUMNS) if download_log_rows \
                else pd.DataFrame(columns=DOWNLOAD_LOG_COLUMNS)
            atomic_write_parquet(log_df, DOWNLOAD_LOG_PATH)
            logger.info(f"checkpoint saved after record {i+1}/{len(plan_indices)} meeting_id={row['meeting_id']}")
        if i < len(plan_indices) - 1:
            time.sleep(REQUEST_INTERVAL_SECONDS)

# final checkpoint regardless of loop having run
atomic_write_parquet(control_registry, CONTROL_REGISTRY_PATH)
log_df = pd.DataFrame(download_log_rows, columns=DOWNLOAD_LOG_COLUMNS) if download_log_rows \
    else pd.DataFrame(columns=DOWNLOAD_LOG_COLUMNS)
atomic_write_parquet(log_df, DOWNLOAD_LOG_PATH)

print(control_registry["download_status"].value_counts(dropna=False))


INFO meeting_id=N053487 attempt=1 status=200 action=validate


INFO checkpoint saved after record 1/42 meeting_id=N053487


INFO meeting_id=N053485 attempt=1 status=200 action=validate


INFO checkpoint saved after record 2/42 meeting_id=N053485


INFO meeting_id=N053483 attempt=1 status=200 action=validate


...

INFO meeting_id=050350 attempt=1 status=200 action=validate


INFO checkpoint saved after record 41/42 meeting_id=050350


INFO meeting_id=050308 attempt=1 status=200 action=validate


INFO checkpoint saved after record 42/42 meeting_id=050308


download_status
downloaded    42
Name: count, dtype: int64


## 09. 수집 결과 품질검사

`downloaded` 행에 대해 local_pdf_path 존재, page_count>=1, sha256 존재, 실제 디스크 파일 존재를 확인하고,
`parse_status`가 이 단계에서 전혀 건드려지지 않았는지도 검사한다.


In [14]:
def run_crawl_quality_checks(df: pd.DataFrame) -> pd.DataFrame:
    checks = []
    def add(name, status, detail=""):
        checks.append({"check": name, "status": status, "detail": str(detail)})

    total = len(df)
    downloaded = df[df["download_status"] == "downloaded"]
    add("TOTAL_ROWS", "INFO", total)
    add("DOWNLOADED_COUNT", "INFO", len(downloaded))
    for st in ["pending", "failed", "invalid_url", "invalid_content"]:
        add(f"STATUS_{st.upper()}_COUNT", "INFO", int((df["download_status"] == st).sum()))

    missing_local_path = int(downloaded["local_pdf_path"].isna().sum())
    add("DOWNLOADED_MISSING_LOCAL_PATH", "PASS" if missing_local_path == 0 else "FAIL", missing_local_path)

    bad_page_count = downloaded[downloaded["page_count"].isna() | (downloaded["page_count"] < 1)]
    add("DOWNLOADED_BAD_PAGE_COUNT", "PASS" if len(bad_page_count) == 0 else "FAIL", len(bad_page_count))

    missing_sha = int(downloaded["sha256"].isna().sum())
    add("DOWNLOADED_MISSING_SHA256", "PASS" if missing_sha == 0 else "FAIL", missing_sha)

    dup_sha_count = int(downloaded["sha256"].dropna().duplicated().sum())
    add("DUPLICATE_SHA256_ACROSS_MEETINGS", "WARN" if dup_sha_count > 0 else "PASS", dup_sha_count)

    missing_on_disk = sum(1 for _, r in downloaded.iterrows() if not (REPO_ROOT / r["local_pdf_path"]).exists())
    add("LOCAL_PDF_PATH_EXISTS_ON_DISK", "PASS" if missing_on_disk == 0 else "FAIL", missing_on_disk)

    unresolved = df[(df["download_status"] == "pending") & df["error_message"].notna()]
    add("UNRESOLVED_PENDING_WITH_ERROR", "INFO", len(unresolved))

    parse_touched = int((df["parse_status"] != "pending").sum())
    add("PARSE_STATUS_UNTOUCHED", "PASS" if parse_touched == 0 else "FAIL", parse_touched)

    return pd.DataFrame(checks)

crawl_quality = run_crawl_quality_checks(control_registry)
atomic_write_parquet(crawl_quality, CRAWL_QUALITY_PATH)
print(crawl_quality.to_string(index=False))


                           check status detail
                      TOTAL_ROWS   INFO     42
                DOWNLOADED_COUNT   INFO     42
            STATUS_PENDING_COUNT   INFO      0
             STATUS_FAILED_COUNT   INFO      0
        STATUS_INVALID_URL_COUNT   INFO      0
    STATUS_INVALID_CONTENT_COUNT   INFO      0
   DOWNLOADED_MISSING_LOCAL_PATH   PASS      0
       DOWNLOADED_BAD_PAGE_COUNT   PASS      0
       DOWNLOADED_MISSING_SHA256   PASS      0
DUPLICATE_SHA256_ACROSS_MEETINGS   PASS      0
   LOCAL_PDF_PATH_EXISTS_ON_DISK   PASS      0
   UNRESOLVED_PENDING_WITH_ERROR   INFO      0
          PARSE_STATUS_UNTOUCHED   PASS      0


## 10. Manifest · Data Dictionary · Crawl Report 생성

`02_pdf_crawler.ipynb` 실행 완료가 곧바로 전체 PDF ETL 완료를 의미하지 않는다. 이 셀의 산출물은
어디까지나 PDF 원본 수집 단계의 증거 자료다.


In [15]:
def compute_source_domains(df: pd.DataFrame) -> list:
    return sorted(set(urlparse(u).netloc for u in df["source_url"].dropna()))

SOURCE_DOMAINS = compute_source_domains(control_registry)

row_counts = {
    "total": int(len(control_registry)),
    "downloaded": int((control_registry["download_status"] == "downloaded").sum()),
    "reused_existing": int(control_registry["reused_existing_file"].sum()),
    "failed": int((control_registry["download_status"] == "failed").sum()),
    "invalid_url": int((control_registry["download_status"] == "invalid_url").sum()),
    "invalid_content": int((control_registry["download_status"] == "invalid_content").sum()),
    "pending": int((control_registry["download_status"] == "pending").sum()),
}

if CRITICAL_GATE_REPORT["status"] == "FAILED" or EXISTING_PDF_CIRCUIT_BREAKER["triggered"]:
    EXECUTION_RESULT_STATUS = "FAILED"
    CRAWL_GATE = "CRAWL_GATE_FAIL"
elif row_counts["failed"] > 0 or row_counts["invalid_url"] > 0 or row_counts["invalid_content"] > 0 or row_counts["pending"] > 0:
    EXECUTION_RESULT_STATUS = "PARTIAL"
    CRAWL_GATE = "CRAWL_GATE_CONDITIONAL_PASS"
else:
    EXECUTION_RESULT_STATUS = "COMPLETE"
    CRAWL_GATE = "CRAWL_GATE_PASS"

manifest = {
    "stage": STAGE,
    "execution_contract_status": EXECUTION_CONTRACT_STATUS,
    "execution_result_status": EXECUTION_RESULT_STATUS,
    "crawl_gate": CRAWL_GATE,
    "run_started_at": RUN_STARTED_AT,
    "run_finished_at": datetime.now(timezone.utc).isoformat(),
    "repo_root": str(REPO_ROOT),
    "git_branch": GIT_BRANCH,
    "git_commit": GIT_COMMIT,
    "input_file": {
        "path": registry_audit["path"],
        "size_bytes": registry_audit["size_bytes"],
        "sha256": registry_audit["sha256"],
        "selected_sheet": SELECTED_SHEET,
    },
    "registry_critical_gate": CRITICAL_GATE_REPORT,
    "existing_pdf_circuit_breaker": EXISTING_PDF_CIRCUIT_BREAKER,
    "row_counts": row_counts,
    "source_domains": SOURCE_DOMAINS,
    "outputs": {
        "control_registry": str(CONTROL_REGISTRY_PATH.relative_to(REPO_ROOT)),
        "download_log": str(DOWNLOAD_LOG_PATH.relative_to(REPO_ROOT)),
        "crawl_quality": str(CRAWL_QUALITY_PATH.relative_to(REPO_ROOT)),
        "log_file": str(LOG_PATH.relative_to(REPO_ROOT)),
    },
    "environment": ENV_INFO,
    "http_policy": {
        "max_workers": MAX_WORKERS,
        "request_interval_seconds": REQUEST_INTERVAL_SECONDS,
        "max_retries": MAX_RETRIES,
        "retry_backoff_seconds": RETRY_BACKOFF_SECONDS,
        "connect_timeout_seconds": CONNECT_TIMEOUT_SECONDS,
        "read_timeout_seconds": READ_TIMEOUT_SECONDS,
        "user_agent": USER_AGENT,
    },
    "not_generated_this_stage": [
        "pages.parquet", "blocks.parquet", "speaker_turns.parquet",
        "retrieval_segments.parquet", "audit_minutes.sqlite",
        "OCR_output", "TFIDF_output", "embedding_output",
    ],
}

with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2, default=str)

print(f"EXECUTION_RESULT_STATUS={EXECUTION_RESULT_STATUS}  CRAWL_GATE={CRAWL_GATE}")
print(json.dumps(row_counts, ensure_ascii=False, indent=2))
print("source_domains:", SOURCE_DOMAINS)


EXECUTION_RESULT_STATUS=COMPLETE  CRAWL_GATE=CRAWL_GATE_PASS
{
  "total": 42,
  "downloaded": 42,
  "reused_existing": 0,
  "failed": 0,
  "invalid_url": 0,
  "invalid_content": 0,
  "pending": 0
}
source_domains: ['record.assembly.go.kr']


In [16]:
data_dict_md = f"""# 02_pdf_crawler Data Dictionary

Stage: `{STAGE}` — PDF 원본 수집/검증 단계의 산출물만 다룬다. 텍스트 추출/OCR/블록/발언자/TF-IDF/embedding 컬럼은 없다.

## control_registry.parquet

Grain: Excel Registry 1행 = 회의 1건.

| 컬럼 | dtype | 설명 |
|---|---|---|
| meeting_id | string (PK) | 원본 회의ID, 정수 변환 금지 |
| meeting_number | string nullable | 원본 `대수`에서 숫자부만 추출 (예: `제22대` → `22`) |
| meeting_count | string nullable | 원본 `회기`에서 숫자부만 추출 (예: `제429회` → `429`) |
| meeting_year | string nullable | 회의일자 연도 마지막 2자리 (YY) |
| meeting_date | string nullable | 회의일자의 MMDD |
| meeting_type | string nullable | 원본 `회의종류` |
| committee_code | string nullable | 원본 `위원회코드` |
| committee_name | string nullable | 원본 `위원회명` |
| source_url | string | Excel의 다운로드 URL. HTTP 요청 대상은 이 컬럼만 사용 |
| source_row_no | Int64 | 원본 Excel 행 번호(헤더 제외, 1-based) |
| source_* | string nullable | 위에 매핑되지 않은 원본 컬럼 보존 (예: `source_차수`) |
| download_status | string | `pending`, `downloaded`, `failed`, `invalid_url`, `invalid_content` |
| http_status | Int64 nullable | 최종 HTTP status code |
| final_url | string nullable | redirect 이후 최종 URL |
| local_pdf_path | string nullable | repo-relative 저장 경로 (`pdf_raw_data/{{meeting_id}}.pdf`) |
| file_size | Int64 nullable | 저장된 PDF 바이트 크기 |
| sha256 | string nullable | PDF 바이너리 SHA-256 |
| page_count | Int64 nullable | `fitz.open().page_count` |
| reused_existing_file | boolean | 기존 유효 PDF를 재사용했는지 여부 |
| error_message | string nullable | 실패/보류 사유 코드 |
| critical_gate_failed | boolean | Registry Critical Gate 실패 행 여부(감사용) |
| parse_status | string | 이 단계에서는 항상 `pending` 고정 |
| parser_name | string nullable | 이 단계에서는 항상 null (파싱 미실행) |
| parser_version | string nullable | 이 단계에서는 항상 null |
| extracted_char_count | Int64 nullable | 이 단계에서는 항상 null |
| ocr_page_count | Int64 nullable | 이 단계에서는 항상 null |

## download_log.parquet

Grain: HTTP 시도(attempt) 1회.

| 컬럼 | dtype | 설명 |
|---|---|---|
| meeting_id | string FK | control_registry.meeting_id |
| attempt_no | Int64 | 1부터 시작하는 시도 순번 |
| timestamp | string (ISO8601 UTC) | 요청 시각 |
| url_requested | string | 실제 요청 URL |
| http_status | Int64 nullable | 응답 status (네트워크 예외 시 null) |
| final_url | string nullable | redirect 최종 URL |
| bytes_received | Int64 | 스트리밍으로 받은 바이트 수 |
| action | string | `validate`/`retry`/`retry_after`/`retry_backoff`/`invalid_url`/`failed` |
| error_message | string nullable | 오류/사유 코드 |
| elapsed_seconds | Float64 | 요청 소요 시간 |
| retry_after_seconds | Float64 nullable | 429 `Retry-After` 값 |

## crawl_quality.parquet

Grain: 품질검사 항목 1개. 컬럼: `check`(str), `status`(INFO/PASS/FAIL/WARN), `detail`(str).

## 생성 규칙 노트

- 모든 결측은 빈 문자열이 아니라 null(`pd.NA`)로 저장한다.
- `meeting_id`/`meeting_number`/`meeting_count`/`meeting_year`/`meeting_date`는 숫자 캐스팅 금지(SSOT.md §4.1).
- 이 문서는 `02_pdf_crawler.ipynb` 실행 시 자동 생성되며 수동 편집분은 다음 실행에서 덮어써진다.
"""

with open(DATA_DICT_PATH, "w", encoding="utf-8") as f:
    f.write(data_dict_md)
print(f"data dictionary written: {DATA_DICT_PATH.relative_to(REPO_ROOT)}")


data dictionary written: data_dict/02_pdf_crawler_data_dictionary.md


In [17]:
def fmt_check_table(df: pd.DataFrame) -> str:
    lines = ["| check | status | detail |", "|---|---|---|"]
    for _, r in df.iterrows():
        lines.append(f"| {r['check']} | {r['status']} | {r['detail']} |")
    return "\n".join(lines)

report_lines = []
report_lines.append(f"# CRAWL_REPORT — {STAGE}\n")
report_lines.append(f"- 실행 시각(UTC): {RUN_STARTED_AT} ~ {manifest['run_finished_at']}")
report_lines.append(f"- git branch / commit: `{GIT_BRANCH}` / `{GIT_COMMIT}`")
report_lines.append(f"- 계약 반영 상태: `{EXECUTION_CONTRACT_STATUS}`")
report_lines.append(f"- 실행 결과 상태: **`{EXECUTION_RESULT_STATUS}`**")
report_lines.append(f"- Crawl Gate: **`{CRAWL_GATE}`**\n")

report_lines.append("## 입력")
report_lines.append(f"- Excel: `{registry_audit['path']}` (sheet=`{SELECTED_SHEET}`, sha256=`{registry_audit['sha256'][:16]}...`)")
report_lines.append(f"- 총 행 수: {row_counts['total']}\n")

report_lines.append("## Registry Critical Gate")
report_lines.append(f"- 상태: **{CRITICAL_GATE_REPORT['status']}**")
report_lines.append(fmt_check_table(pd.DataFrame(CRITICAL_GATE_REPORT["checks"]).rename(
    columns={"failed_count": "status"})) if CRITICAL_GATE_REPORT["checks"] else "- (검사 항목 없음)")
report_lines.append("")

report_lines.append("## 기존 PDF Circuit Breaker")
report_lines.append(f"- triggered: {EXISTING_PDF_CIRCUIT_BREAKER['triggered']}")
if EXISTING_PDF_CIRCUIT_BREAKER["triggered"]:
    report_lines.append(f"- meeting_id: {EXISTING_PDF_CIRCUIT_BREAKER['meeting_id']}")
    report_lines.append(f"- error: {EXISTING_PDF_CIRCUIT_BREAKER['error_message']}")
report_lines.append("")

report_lines.append("## 다운로드 결과")
for k, v in row_counts.items():
    report_lines.append(f"- {k}: {v}")
report_lines.append("")

report_lines.append("## 원천 도메인 (source_url 사후 집계)")
for d in SOURCE_DOMAINS:
    report_lines.append(f"- {d}")
report_lines.append("")

report_lines.append("## 수집 결과 품질검사")
report_lines.append(fmt_check_table(crawl_quality))
report_lines.append("")

report_lines.append("## 의존성")
for k, v in ENV_INFO.items():
    report_lines.append(f"- {k}: {v}")
report_lines.append("")

report_lines.append("## 이 단계 산출물")
for k, v in manifest["outputs"].items():
    report_lines.append(f"- {k}: `{v}`")
report_lines.append(f"- data_dict: `{DATA_DICT_PATH.relative_to(REPO_ROOT)}`")
report_lines.append("")

report_lines.append("## 이 단계가 의미하지 않는 것")
report_lines.append(
    "PDF 본문 텍스트 추출 성공, OCR 성공, 페이지/블록 분할, 발언자 판별, "
    "speaker turn 생성, TF-IDF 검색 — 어느 것도 이 단계의 성공 기준에 포함되지 않는다."
)

CRAWL_REPORT_TEXT = "\n".join(report_lines)
with open(CRAWL_REPORT_PATH, "w", encoding="utf-8") as f:
    f.write(CRAWL_REPORT_TEXT)

print(CRAWL_REPORT_TEXT)


# CRAWL_REPORT — PDF_CRAWL_AND_DOWNLOAD

- 실행 시각(UTC): 2026-07-21T05:25:45.779447+00:00 ~ 2026-07-21T05:27:18.323946+00:00
- git branch / commit: `P3_DATA_RAW` / `236ad55872afe97df3c9308f9fbd3535f5d287be`
- 계약 반영 상태: `CRAWLER_EXECUTION_CONTRACT_ACTIVE`
- 실행 결과 상태: **`COMPLETE`**
- Crawl Gate: **`CRAWL_GATE_PASS`**

## 입력
- Excel: `data_origin/국정감사회의록_문화체육 2020~.xlsx` (sheet=`Sheet1`, sha256=`a4e7bb5ae119556e...`)
- 총 행 수: 42

## Registry Critical Gate
- 상태: **PASSED**
| check | status | detail |
|---|---|---|
| MISSING_MEETING_ID | 0 |  |
| DUPLICATE_MEETING_ID_PATH_COLLISION | 0 |  |
| MISSING_SOURCE_URL | 0 |  |
| NON_HTTP_SCHEME_URL | 0 |  |
| UNSAFE_MEETING_ID_FILENAME | 0 |  |
| EXPECTED_PDF_PATH_COLLISION | 0 |  |
| DUPLICATE_SOURCE_URL_ACROSS_MEETING_IDS | 0 | row-level pending, not a hard gate failure |

## 기존 PDF Circuit Breaker
- triggered: False

## 다운로드 결과
- total: 42
- downloaded: 42
- reused_existing: 0
- failed: 0
- invalid_url: 0
- invalid_content: 0
- pending: 0

## 원천

In [18]:
print("STAGE:", STAGE)
print("EXECUTION_STATUS:", EXECUTION_RESULT_STATUS)
print("TEXT_EXTRACTION: NOT_RUN")
print("OCR: NOT_RUN")
print("BLOCK_PARSING: NOT_RUN")
print("SPEAKER_TURN: NOT_RUN")
print("TFIDF: NOT_RUN")
print("EMBEDDING: NOT_RUN")
print("SQLITE: NOT_RUN")
print("CRAWL_GATE:", CRAWL_GATE)


STAGE: PDF_CRAWL_AND_DOWNLOAD
EXECUTION_STATUS: COMPLETE
TEXT_EXTRACTION: NOT_RUN
OCR: NOT_RUN
BLOCK_PARSING: NOT_RUN
SPEAKER_TURN: NOT_RUN
TFIDF: NOT_RUN
EMBEDDING: NOT_RUN
SQLITE: NOT_RUN
CRAWL_GATE: CRAWL_GATE_PASS
